In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
project_root = Path('.').resolve().parent
ref_dir = project_root/'data'/'reference'
drift_dir = project_root/'data'/'drift'

In [4]:
# Sample of the training distribution

housing = pd.read_parquet(project_root/'data'/'processed'/'london_housing_2008_24.parquet')
pc = pd.read_parquet(project_root /'data'/'supplementary'/'postcodes_london.parquet')
train = housing[housing['year'] <= 2023]

In [5]:
train.shape

(1371211, 18)

In [6]:
reference = reference = train.merge(pc, on='postcode', how='left')

In [7]:
reference.shape

(1371211, 24)

In [8]:
keep = ['price', 'propertytype', 'duration', 'lat', 'lon', 'zone',
        'imd_index', 'avg_income', 'dist_station']
reference = reference[keep]

In [9]:
reference.shape

(1371211, 9)

In [10]:
reference.to_parquet(ref_dir/'training_reference.parquet', index=False)

In [14]:
# New Data

pp_columns = ['transactionid', 'price', 'date', 'postcode', 'propertytype',
               'old_new', 'duration', 'paon', 'saon', 'street', 'locality',
               'town', 'district', 'county', 'ppd_cat', 'status']

In [15]:
new_pp = pd.read_csv(project_root/'data'/'supplementary'/'pp-2025.csv',
                     header=None, names=pp_columns, usecols=['price', 'postcode', 'propertytype', 'duration', 'county'])

In [16]:
new_pp.head()

,price,postcode,propertytype,duration,county
0,560000,PE1 2QU,D,F,CITY OF PETERBOROUGH
1,230000,PE2 9RY,T,F,CITY OF PETERBOROUGH
2,272000,PE16 6BL,D,F,CAMBRIDGESHIRE
3,249950,CB4 3RY,T,L,CAMBRIDGESHIRE
4,650000,CB1 3BH,T,F,CAMBRIDGESHIRE


In [17]:
new_pp.shape

(802761, 5)

In [18]:
new_pp = new_pp[new_pp['county'] == 'GREATER LONDON']
new_pp = new_pp[new_pp['propertytype'].isin(['D', 'F', 'S', 'T'])]
new_pp = new_pp[['price', 'postcode', 'propertytype', 'duration']]

In [19]:
new_pp.head()

,price,postcode,propertytype,duration
192,200000,IG6 2DZ,F,L
193,420000,E2 7EL,F,L
194,662000,IG1 4LB,T,F
195,184000,RM1 1AR,F,L
196,575000,E14 0JU,F,L


In [20]:
new_pp.shape

(84310, 4)

In [21]:
new_pp.to_parquet(drift_dir/'london_2025.parquet', index=False)